In [1]:
from google.colab import drive
import torch

drive.mount("/content/drive")

print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")

if torch.cuda.is_available():
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

Mounted at /content/drive
GPU: CPU


In [1]:
from google.colab import drive
from pathlib import Path
import torch

drive.mount("/content/drive")

print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")

folder = Path("/content/drive/MyDrive/NMT_Sports_Project/data/splits")

for split in ["train", "validation", "test"]:
    path = folder / f"{split}.parquet"
    print(split, "✅ موجود است" if path.is_file() else "❌ پیدا نشد")

Mounted at /content/drive
GPU: NVIDIA L4
train ✅ موجود است
validation ✅ موجود است
test ✅ موجود است


In [2]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/NMT_Sports_Project")

train_df = pd.read_parquet(BASE / "data/splits/train.parquet")
val_df = pd.read_parquet(BASE / "data/splits/validation.parquet")
test_df = pd.read_parquet(BASE / "data/splits/test.parquet")

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(f"{name}: {len(df)} rows")

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nProcessed data files:")
folder = BASE / "data/processed"
found = False
if folder.exists():
    for p in sorted(folder.rglob("*")):
        if p.is_file() and p.suffix.lower() in {
            ".csv", ".parquet", ".json", ".jsonl", ".zip"
        }:
            print(p.relative_to(BASE))
            found = True
if not found:
    print("No processed data files found.")

Train: 1023 rows
Validation: 150 rows
Test: 150 rows

Train columns:
['sample_id', 'source', 'target', 'context_1_text', 'context_2_text', 'context_3_text', 'context_group_id', 'context_1_contains_source', 'context_2_contains_source', 'context_3_contains_source', 'source_in_any_context', 'source_context_overlap_count', 'context_1_segments_json', 'context_2_segments_json', 'context_3_segments_json']

Processed data files:
data/processed/domain_tokenized_full_v1/dataset_dict.json
data/processed/domain_tokenized_full_v1/train/dataset_info.json
data/processed/domain_tokenized_full_v1/train/state.json
data/processed/domain_tokenized_full_v1/validation/dataset_info.json
data/processed/domain_tokenized_full_v1/validation/state.json
data/processed/domain_train_augmented.parquet
data/processed/domain_validation_normalized.parquet


In [3]:
from pathlib import Path
import pandas as pd
import json

BASE = Path("/content/drive/MyDrive/NMT_Sports_Project")
processed = BASE / "data/processed"

train_aug = pd.read_parquet(
    processed / "domain_train_augmented.parquet"
)
val_norm = pd.read_parquet(
    processed / "domain_validation_normalized.parquet"
)

for name, df in [
    ("Augmented train", train_aug),
    ("Normalized validation", val_norm),
]:
    print(f"\n{name}: {len(df)} rows")
    print("Columns:", df.columns.tolist())
    print("Missing values:", int(df.isna().sum().sum()))

tokenized = processed / "domain_tokenized_full_v1"
for split in ["train", "validation"]:
    folder = tokenized / split
    print(f"\nTokenized {split} files:")
    print([p.name for p in folder.iterdir() if p.is_file()])

    info_path = folder / "dataset_info.json"
    if info_path.exists():
        info = json.loads(info_path.read_text())
        print("Features:", info.get("features"))


Augmented train: 1785 rows
Columns: ['source', 'target', 'variant']
Missing values: 0

Normalized validation: 150 rows
Columns: ['source', 'target']
Missing values: 0

Tokenized train files:
['data-00000-of-00001.arrow', 'state.json', 'dataset_info.json']
Features: {'input_ids': {'feature': {'dtype': 'int32', '_type': 'Value'}, '_type': 'List'}, 'attention_mask': {'feature': {'dtype': 'int8', '_type': 'Value'}, '_type': 'List'}, 'labels': {'feature': {'dtype': 'int64', '_type': 'Value'}, '_type': 'List'}}

Tokenized validation files:
['data-00000-of-00001.arrow', 'state.json', 'dataset_info.json']
Features: {'input_ids': {'feature': {'dtype': 'int32', '_type': 'Value'}, '_type': 'List'}, 'attention_mask': {'feature': {'dtype': 'int8', '_type': 'Value'}, '_type': 'List'}, 'labels': {'feature': {'dtype': 'int64', '_type': 'Value'}, '_type': 'List'}}


In [4]:
%pip -q install "transformers==4.57.1" datasets accelerate sentencepiece sacrebleu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 141.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [5]:
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from datasets import load_from_disk
import accelerate
import sacrebleu

print("Transformers:", transformers.__version__)
print("✅ Training imports OK")

Transformers: 4.57.1
✅ Training imports OK


In [6]:
import os, json, time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import sacrebleu
import transformers
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
    set_seed,
)

assert torch.cuda.is_available(), "GPU is not connected."
set_seed(42)

BASE = Path("/content/drive/MyDrive/NMT_Sports_Project")
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
OUT = BASE / "models" / f"Model2_epoch_extension_NEW_{stamp}"
OUT.mkdir(parents=True, exist_ok=False)

MODEL_ID = "facebook/nllb-200-distilled-600M"

data = load_from_disk(
    str(BASE / "data/processed/domain_tokenized_full_v1")
)
assert len(data["train"]) == 1785
assert len(data["validation"]) == 150

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    src_lang="pes_Arab",
    tgt_lang="eng_Latn",
)

# Check compatibility of stored token IDs.
for split in ["train", "validation"]:
    for row in data[split]:
        for key in ["input_ids", "labels"]:
            ids = [int(x) for x in row[key] if int(x) != -100]
            assert ids and min(ids) >= 0 and max(ids) < len(tokenizer)

print("Training examples:", len(data["train"]))
print("Validation examples:", len(data["validation"]))
print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUT)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

model.generation_config.forced_bos_token_id = (
    tokenizer.convert_tokens_to_ids("eng_Latn")
)
model.generation_config.num_beams = 4
model.generation_config.max_length = 128

# Passing model here prepares decoder inputs correctly
# when label smoothing is enabled.
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

bleu_metric = sacrebleu.metrics.BLEU()
chrf_metric = sacrebleu.metrics.CHRF(word_order=2)
evaluation_number = 0

def compute_metrics(eval_prediction):
    global evaluation_number
    predictions, labels = eval_prediction
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.where(
        predictions != -100, predictions, tokenizer.pad_token_id
    )
    labels = np.where(
        labels != -100, labels, tokenizer.pad_token_id
    )

    hypotheses = tokenizer.batch_decode(
        predictions, skip_special_tokens=True
    )
    references = tokenizer.batch_decode(
        labels, skip_special_tokens=True
    )

    bleu = bleu_metric.corpus_score(hypotheses, [references]).score
    chrf = chrf_metric.corpus_score(hypotheses, [references]).score

    evaluation_number += 1
    pd.DataFrame({
        "validation_position": range(len(hypotheses)),
        "decoded_reference": references,
        "prediction": hypotheses,
    }).to_csv(
        OUT / f"validation_eval_{evaluation_number:02d}.csv",
        index=False,
    )

    return {"bleu": bleu, "chrfpp": chrf}


class SaveEvidence(TrainerCallback):
    def on_evaluate(self, args, state, control, **kwargs):
        pd.DataFrame(state.log_history).to_csv(
            OUT / "training_history.csv", index=False
        )
        state.save_to_json(str(OUT / "training_state.json"))

    def on_save(self, args, state, control, model=None, **kwargs):
        # Preserve epoch 3 for the later controlled comparison.
        if state.epoch is not None and abs(state.epoch - 3.0) < 0.01:
            destination = OUT / "epoch_3_weights"
            model.save_pretrained(destination)
            tokenizer.save_pretrained(destination)


args = Seq2SeqTrainingArguments(
    output_dir=str(OUT / "checkpoints"),
    num_train_epochs=8,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,  # effective batch = 32
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    warmup_ratio=0.10,
    label_smoothing_factor=0.10,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    seed=42,
    data_seed=42,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    predict_with_generate=True,
    generation_num_beams=4,
    generation_max_length=128,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    save_total_limit=1,
    save_only_model=True,
    eval_accumulation_steps=1,
    report_to="none",
)

protocol = {
    "run_type": "new_training_run_not_resume",
    "model": MODEL_ID,
    "seed": 42,
    "epochs": 8,
    "early_stopping": False,
    "selection": "highest validation BLEU",
    "train_rows": len(data["train"]),
    "validation_rows": len(data["validation"]),
    "effective_batch_size": 32,
    "generation_max_length": 128,
    "generation_num_beams": 4,
    "transformers_version": transformers.__version__,
    "test_used": False,
    "checkpoint_contains_optimizer": False,
}
(OUT / "protocol.json").write_text(
    json.dumps(protocol, indent=2), encoding="utf-8"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[SaveEvidence()],
)

start = time.time()
trainer.train()

pd.DataFrame(trainer.state.log_history).to_csv(
    OUT / "training_history.csv", index=False
)

summary = {
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_bleu": trainer.state.best_metric,
    "elapsed_seconds": time.time() - start,
    "bleu_signature": str(bleu_metric.get_signature()),
    "chrfpp_signature": str(chrf_metric.get_signature()),
}
(OUT / "summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)

print("\nTRAINING FINISHED")
print(json.dumps(summary, indent=2))
print("Saved to:", OUT)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

ValueError: You cannot specify both decoder_input_ids and decoder_inputs_embeds at the same time

In [11]:
import json
import time
import torch
import pandas as pd
from transformers import DataCollatorForSeq2Seq, set_seed

assert trainer.state.global_step == 0, (
    "Some training steps have already completed; report this before restarting."
)

base_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

def fixed_collator(features):
    batch = base_collator(features)
    labels = batch["labels"]

    # Standard decoder right shift for this NLLB/M2M100 model.
    decoder_ids = labels.new_full(
        labels.shape, model.config.pad_token_id
    )
    decoder_ids[:, 1:] = labels[:, :-1].clone()
    decoder_ids[:, 0] = model.config.decoder_start_token_id
    decoder_ids.masked_fill_(
        decoder_ids == -100, model.config.pad_token_id
    )

    batch["decoder_input_ids"] = decoder_ids
    return batch

trainer.data_collator = fixed_collator

# Check one example before starting the full run.
probe = fixed_collator([data["train"][0]])
probe = {k: v.to(trainer.args.device) for k, v in probe.items()}
model.eval()

with torch.no_grad(), trainer.compute_loss_context_manager():
    loss = trainer.compute_loss(model, dict(probe))

assert torch.isfinite(loss).item(), "Non-finite loss."
print("✅ Decoder fixed. Initial loss:", float(loss))
del probe, loss

protocol["decoder_input_preparation"] = "explicit right shift"
(OUT / "protocol.json").write_text(
    json.dumps(protocol, indent=2), encoding="utf-8"
)

set_seed(42)
model.train()
start = time.time()
trainer.train()

pd.DataFrame(trainer.state.log_history).to_csv(
    OUT / "training_history.csv", index=False
)
trainer.state.save_to_json(str(OUT / "training_state.json"))

summary = {
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_bleu": trainer.state.best_metric,
    "elapsed_seconds": time.time() - start,
    "bleu_signature": str(bleu_metric.get_signature()),
    "chrfpp_signature": str(chrf_metric.get_signature()),
}
(OUT / "summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)

print("TRAINING FINISHED")
print(json.dumps(summary, indent=2))
print("Saved to:", OUT)

AssertionError: Some training steps have already completed; report this before restarting.

In [1]:
from pathlib import Path
from collections import Counter
import pandas as pd
import unicodedata
import re

BASE = Path("/content/drive/MyDrive/NMT_Sports_Project")

aug = pd.read_parquet(
    BASE / "data/processed/domain_train_augmented.parquet"
)
valid = pd.read_parquet(
    BASE / "data/processed/domain_validation_normalized.parquet"
)
raw_val = pd.read_parquet(BASE / "data/splits/validation.parquet")
raw_test = pd.read_parquet(BASE / "data/splits/test.parquet")

def norm(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(str.maketrans({"ي": "ی", "ك": "ک"}))
    text = text.replace("\u200c", " ").replace("\u200d", "")
    return re.sub(r"\s+", " ", text).strip().lower()

def clean_ids(ids):
    return [
        int(x) for x in ids
        if int(x) != -100 and int(x) != tokenizer.pad_token_id
    ]

checks = []

# Retokenize each original text without truncation and compare IDs.
# Any mismatch needs inspection; it does not automatically mean leakage.
for split, frame in [("train", aug), ("validation", valid)]:
    saved = data[split]
    assert len(saved) == len(frame), f"Row count mismatch: {split}"

    sources = tokenizer(
        frame["source"].astype(str).tolist(),
        truncation=False, padding=False
    )["input_ids"]
    targets = tokenizer(
        text_target=frame["target"].astype(str).tolist(),
        truncation=False, padding=False
    )["input_ids"]

    source_bad = sum(
        clean_ids(a) != clean_ids(b)
        for a, b in zip(sources, saved["input_ids"])
    )
    target_bad = sum(
        clean_ids(a) != clean_ids(b)
        for a, b in zip(targets, saved["labels"])
    )
    print(f"{split}: source mismatches={source_bad}, "
          f"target mismatches={target_bad}")
    checks.append(source_bad == 0 and target_bad == 0)

def pairs(frame):
    return Counter(
        (norm(s), norm(t))
        for s, t in zip(frame["source"], frame["target"])
    )

same_validation = pairs(valid) == pairs(raw_val)
print("Validation matches original split:", same_validation)
checks.append(same_validation)

for name, heldout in [("Validation", raw_val), ("Test", raw_test)]:
    for col in ["source", "target"]:
        overlap = (
            set(aug[col].map(norm))
            & set(heldout[col].map(norm))
        )
        print(f"Augmented train / {name} {col} overlap:", len(overlap))
        checks.append(len(overlap) == 0)

print("\nBest checkpoint:", trainer.state.best_model_checkpoint)
print("✅ Checks passed" if all(checks)
      else "⚠️ Inspect mismatches before Test")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/NMT_Sports_Project/data/processed/domain_train_augmented.parquet'

In [2]:
from google.colab import drive
from pathlib import Path
import torch

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/NMT_Sports_Project")
RUN = BASE / "models/Model2_epoch_extension_NEW_20260923_135303"

paths = {
    "Project": BASE,
    "Augmented train":
        BASE / "data/processed/domain_train_augmented.parquet",
    "Tokenized data":
        BASE / "data/processed/domain_tokenized_full_v1",
    "Training summary": RUN / "summary.json",
    "Best checkpoint": RUN / "checkpoints/checkpoint-448",
    "Epoch 3 weights": RUN / "epoch_3_weights",
}

for name, path in paths.items():
    print(name, "✅" if path.exists() else "❌")

print("\nGPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")
print("Previous variables available:",
      all(name in globals() for name in ["tokenizer", "data", "trainer"]))

Mounted at /content/drive
Project ✅
Augmented train ✅
Tokenized data ✅
Training summary ❌
Best checkpoint ❌
Epoch 3 weights ✅

GPU: NVIDIA L4
Previous variables available: False


In [3]:
from pathlib import Path
import json

BASE = Path("/content/drive/MyDrive/NMT_Sports_Project")
runs = sorted((BASE / "models").glob("Model2_epoch_extension_NEW*"))

for run in runs:
    print("\nRUN:", run.name)

    summary = run / "summary.json"
    if summary.exists():
        print("SUMMARY:", summary.read_text())

    for folder in sorted(run.iterdir()):
        if folder.is_dir():
            print("FOLDER:", folder.name)

    for pattern in [
        "**/model.safetensors",
        "**/model-*.safetensors",
        "**/pytorch_model.bin",
        "**/trainer_state.json",
        "**/training_state.json",
        "**/training_history.csv",
    ]:
        for path in sorted(run.glob(pattern)):
            print(
                path.relative_to(run),
                f"({path.stat().st_size / 1024**2:.1f} MB)"
            )
            if path.name in {"trainer_state.json", "training_state.json"}:
                state = json.loads(path.read_text())
                print("  Epoch:", state.get("epoch"))
                print("  Best:", state.get("best_model_checkpoint"))


RUN: Model2_epoch_extension_NEW_20260923_135303
FOLDER: checkpoints
FOLDER: epoch_3_weights
checkpoints/checkpoint-168/model.safetensors (2346.4 MB)
epoch_3_weights/model.safetensors (2346.4 MB)
checkpoints/checkpoint-168/trainer_state.json (0.0 MB)
  Epoch: 3.0
  Best: /content/drive/MyDrive/NMT_Sports_Project/models/Model2_epoch_extension_NEW_20260923_135303/checkpoints/checkpoint-168
checkpoints/checkpoint-224/trainer_state.json (0.0 MB)
  Epoch: 4.0
  Best: /content/drive/MyDrive/NMT_Sports_Project/models/Model2_epoch_extension_NEW_20260923_135303/checkpoints/checkpoint-224
training_state.json (0.0 MB)
  Epoch: 4.0
  Best: /content/drive/MyDrive/NMT_Sports_Project/models/Model2_epoch_extension_NEW_20260923_135303/checkpoints/checkpoint-168
training_history.csv (0.0 MB)


In [4]:
from pathlib import Path

RUN = Path(
    "/content/drive/MyDrive/NMT_Sports_Project/models/"
    "Model2_epoch_extension_NEW_20260923_135303"
)

print("FILES:")
for p in sorted(RUN.rglob("*")):
    if p.is_file():
        print(
            p.relative_to(RUN),
            f"| {p.stat().st_size:,} bytes"
        )

history = RUN / "training_history.csv"
if history.exists():
    print("\nTRAINING HISTORY:")
    print(history.read_text()[-6000:])

FILES:
checkpoints/checkpoint-168/config.json | 836 bytes
checkpoints/checkpoint-168/generation_config.json | 215 bytes
checkpoints/checkpoint-168/model.safetensors | 2,460,354,912 bytes
checkpoints/checkpoint-168/sentencepiece.bpe.model | 4,852,054 bytes
checkpoints/checkpoint-168/special_tokens_map.json | 4,232 bytes
checkpoints/checkpoint-168/tokenizer.json | 32,240,136 bytes
checkpoints/checkpoint-168/tokenizer_config.json | 39,897 bytes
checkpoints/checkpoint-168/trainer_state.json | 4,486 bytes
checkpoints/checkpoint-168/training_args.bin | 6,161 bytes
checkpoints/checkpoint-224/config.json | 836 bytes
checkpoints/checkpoint-224/generation_config.json | 215 bytes
checkpoints/checkpoint-224/sentencepiece.bpe.model | 4,852,054 bytes
checkpoints/checkpoint-224/special_tokens_map.json | 4,232 bytes
checkpoints/checkpoint-224/tokenizer.json | 32,240,136 bytes
checkpoints/checkpoint-224/tokenizer_config.json | 39,897 bytes
checkpoints/checkpoint-224/trainer_state.json | 5,802 bytes
che

In [5]:
from pathlib import Path
import pandas as pd
import re, unicodedata
from collections import Counter

BASE = Path("/content/drive/MyDrive/NMT_Sports_Project")
RUN = BASE / "models/Model2_epoch_extension_NEW_20260923_135303"

aug = pd.read_parquet(
    BASE / "data/processed/domain_train_augmented.parquet"
)
val = pd.read_parquet(
    BASE / "data/processed/domain_validation_normalized.parquet"
)
raw_val = pd.read_parquet(BASE / "data/splits/validation.parquet")
test = pd.read_parquet(BASE / "data/splits/test.parquet")
pred = pd.read_csv(RUN / "validation_eval_03.csv")

def norm(x):
    x = unicodedata.normalize("NFKC", str(x))
    x = x.translate(str.maketrans({"ي": "ی", "ك": "ک"}))
    x = x.replace("\u200c", " ").replace("\u200d", "")
    return re.sub(r"\s+", " ", x).strip().lower()

def pairs(df):
    return Counter(
        (norm(s), norm(t))
        for s, t in zip(df["source"], df["target"])
    )

print("Validation matches original split:", pairs(val) == pairs(raw_val))
print("Saved prediction rows:", len(pred))

assert len(pred) == len(val), "Validation row counts differ."
print("Decoded reference mismatches:", sum(
    norm(a) != norm(b)
    for a, b in zip(pred["decoded_reference"], val["target"])
))

for name, heldout in [("Validation", raw_val), ("Test", test)]:
    for col in ["source", "target"]:
        overlap = set(aug[col].map(norm)) & set(heldout[col].map(norm))
        print(f"Train / {name} {col} overlap:", len(overlap))

print("\nThree validation examples:")
print(pred[["decoded_reference", "prediction"]].head(3).to_string(index=False))

Validation matches original split: False
Saved prediction rows: 150
Decoded reference mismatches: 16
Train / Validation source overlap: 0
Train / Validation target overlap: 0
Train / Test source overlap: 0
Train / Test target overlap: 0

Three validation examples:
                                                                 decoded_reference                                                                          prediction
                                      Post-game analysis helps improve strategies.                                        Game analysis then helps improve strategies.
Fair play and human dignity are the fundamental principles of professional sports. Fair play and human dignity are the foundational principles of professional sports.
                                            The Hall of Fame honored the champion.                                              The Hall of Fame honored the champion.


In [6]:
from collections import Counter
import pandas as pd

# Are the same sources present, possibly in a different order?
a = val["source"].map(norm).tolist()
b = raw_val["source"].map(norm).tolist()

print("Same sources, ignoring order:", Counter(a) == Counter(b))
print("Same source order:", a == b)

# Compare targets after matching by normalized source.
left = val[["source", "target"]].copy()
right = raw_val[["source", "target"]].copy()
left["key"] = left["source"].map(norm)
right["key"] = right["source"].map(norm)

if left["key"].duplicated().any() or right["key"].duplicated().any():
    print("Duplicate normalized sources: matching needs inspection.")
else:
    joined = left.merge(
        right, on="key", how="outer",
        suffixes=("_processed", "_original"),
        indicator=True, validate="one_to_one"
    )
    changed = joined[
        (joined["_merge"] != "both")
        | (
            joined["target_processed"].map(norm)
            != joined["target_original"].map(norm)
        )
    ]
    print("\nProcessed versus original differences:", len(changed))
    print(changed[
        ["source_processed", "source_original",
         "target_processed", "target_original", "_merge"]
    ].head(10).to_string(index=False))

print("\nDecoded versus processed references:")
count = 0
for i, (decoded, target) in enumerate(
    zip(pred["decoded_reference"], val["target"])
):
    if norm(decoded) != norm(target):
        print(f"\nRow {i}")
        print("Processed:", repr(target))
        print("Decoded  :", repr(decoded))
        count += 1
        if count == 8:
            break

Same sources, ignoring order: False
Same source order: False

Processed versus original differences: 104
                                               source_processed                                                 source_original                                                                                       target_processed                                                                                        target_original     _merge
                                  آمار بازی نشان از برتری داشت.                                                             NaN                                                                   Match statistics showed superiority.                                                                                                    NaN  left_only
                 آمار بازی کمک می کند تا نقاط ضعف شناسایی شوند.                                                             NaN                                                             Match statistics 

In [7]:
import re
import unicodedata
from collections import Counter
from sacrebleu.metrics import BLEU, CHRF

def fa_match(x):
    x = unicodedata.normalize("NFKC", str(x))
    x = x.translate(str.maketrans({"ي": "ی", "ك": "ک"}))
    x = re.sub(r"[\u200c\u200d\u200e\u200f]", " ", x)
    x = re.sub(r"[-‐‑‒–—−]", " ", x)
    return re.sub(r"\s+", " ", x).strip()

processed_keys = val["source"].map(fa_match).tolist()
original_keys = raw_val["source"].map(fa_match).tolist()

same_samples = Counter(processed_keys) == Counter(original_keys)
unique_keys = (
    len(set(processed_keys)) == len(processed_keys)
    and len(set(original_keys)) == len(original_keys)
)

print("Same validation sources after spacing/hyphen normalization:",
      same_samples)
print("Unique matching keys:", unique_keys)

for name, heldout in [("Validation", raw_val), ("Test", test)]:
    overlap = (
        set(aug["source"].map(fa_match))
        & set(heldout["source"].map(fa_match))
    )
    print(f"Normalized train / {name} source overlap:", len(overlap))

if same_samples and unique_keys and len(pred) == len(val):
    original_targets = dict(zip(original_keys, raw_val["target"]))
    refs = [str(original_targets[k]) for k in processed_keys]
    hypotheses = pred["prediction"].fillna("").astype(str).tolist()

    print("Processed/original target differences:",
          sum(norm(a) != norm(b)
              for a, b in zip(val["target"], refs)))

    bleu = BLEU()
    chrf = CHRF(word_order=2)

    print("\nNEW RUN — EPOCH 3 — VALIDATION")
    print("Scored against original English references")
    print("BLEU:", round(bleu.corpus_score(hypotheses, [refs]).score, 4))
    print("chrF++:", round(chrf.corpus_score(hypotheses, [refs]).score, 4))
else:
    print("\nSTOP: source alignment still needs inspection.")

ModuleNotFoundError: No module named 'sacrebleu'

In [8]:
%pip -q install "sacrebleu==2.6.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.3 MB/s eta 0:00:00


In [9]:
import re
import unicodedata
from collections import Counter
from sacrebleu.metrics import BLEU, CHRF

def fa_match(x):
    x = unicodedata.normalize("NFKC", str(x))
    x = x.translate(str.maketrans({"ي": "ی", "ك": "ک"}))
    x = re.sub(r"[\u200c\u200d\u200e\u200f]", " ", x)
    x = re.sub(r"[-‐‑‒–—−]", " ", x)
    return re.sub(r"\s+", " ", x).strip()

processed_keys = val["source"].map(fa_match).tolist()
original_keys = raw_val["source"].map(fa_match).tolist()

same_samples = Counter(processed_keys) == Counter(original_keys)
unique_keys = (
    len(set(processed_keys)) == len(processed_keys)
    and len(set(original_keys)) == len(original_keys)
)

print("Same validation sources after spacing/hyphen normalization:",
      same_samples)
print("Unique matching keys:", unique_keys)

for name, heldout in [("Validation", raw_val), ("Test", test)]:
    overlap = (
        set(aug["source"].map(fa_match))
        & set(heldout["source"].map(fa_match))
    )
    print(f"Normalized train / {name} source overlap:", len(overlap))

if same_samples and unique_keys and len(pred) == len(val):
    original_targets = dict(zip(original_keys, raw_val["target"]))
    refs = [str(original_targets[k]) for k in processed_keys]
    hypotheses = pred["prediction"].fillna("").astype(str).tolist()

    print("Processed/original target differences:",
          sum(norm(a) != norm(b)
              for a, b in zip(val["target"], refs)))

    bleu = BLEU()
    chrf = CHRF(word_order=2)

    print("\nNEW RUN — EPOCH 3 — VALIDATION")
    print("Scored against original English references")
    print("BLEU:", round(bleu.corpus_score(hypotheses, [refs]).score, 4))
    print("chrF++:", round(chrf.corpus_score(hypotheses, [refs]).score, 4))
else:
    print("\nSTOP: source alignment still needs inspection.")

Same validation sources after spacing/hyphen normalization: True
Unique matching keys: True
Normalized train / Validation source overlap: 0
Normalized train / Test source overlap: 0
Processed/original target differences: 0

NEW RUN — EPOCH 3 — VALIDATION
Scored against original English references
BLEU: 49.9842
chrF++: 69.8672


In [12]:
%pip -q install "transformers==4.57.1" sentencepiece "sacrebleu==2.6.0"

In [13]:
from pathlib import Path
import json, time
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu.metrics import BLEU, CHRF

BASE = Path("/content/drive/MyDrive/NMT_Sports_Project")
RUN = BASE / "models/Model2_epoch_extension_NEW_20260923_135303"
CKPT = RUN / "epoch_3_weights"

assert (CKPT / "model.safetensors").is_file()
assert torch.cuda.is_available(), "GPU is not connected."

test_data = pd.read_parquet(BASE / "data/splits/test.parquet")
assert len(test_data) == 150
assert test_data[["source", "target"]].notna().all().all()

tok = AutoTokenizer.from_pretrained(
    CKPT,
    src_lang="pes_Arab",
    tgt_lang="eng_Latn",
    local_files_only=True,
)
net = AutoModelForSeq2SeqLM.from_pretrained(
    CKPT, local_files_only=True
).to("cuda")
net.eval()

# Fixed decoding rule, chosen before viewing Test scores.
# Match the new run's validation generation length.
net.generation_config.max_new_tokens = None

sources = test_data["source"].astype(str).tolist()
references = test_data["target"].astype(str).tolist()
predictions = []

torch.cuda.synchronize()
started = time.time()

with torch.inference_mode():
    for start in range(0, len(sources), 4):
        inputs = tok(
            sources[start:start + 4],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to("cuda")

        generated = net.generate(
            **inputs,
            forced_bos_token_id=tok.convert_tokens_to_ids("eng_Latn"),
            num_beams=4,
            do_sample=False,
            max_length=128,
            use_cache=True,
        )
        predictions.extend(
            tok.batch_decode(generated, skip_special_tokens=True)
        )
        print(f"\rTranslated {len(predictions)}/150", end="", flush=True)

torch.cuda.synchronize()
elapsed = time.time() - started

bleu = BLEU()
chrf = CHRF(word_order=2)
result = {
    "system": "New Model2 run — recovered epoch 3",
    "split": "test",
    "checkpoint": str(CKPT),
    "BLEU": bleu.corpus_score(predictions, [references]).score,
    "chrF++": chrf.corpus_score(predictions, [references]).score,
    "generation_seconds": elapsed,
    "num_beams": 4,
    "max_length": 128,
    "input_max_length": 512,
    "bleu_signature": str(bleu.get_signature()),
    "chrf_signature": str(chrf.get_signature()),
}

# Save only small text outputs, not model weights.
stamp = time.strftime("%Y%m%d_%H%M%S")
output = test_data.copy()
output["prediction"] = predictions
output.to_csv(
    RUN / f"epoch3_test_predictions_{stamp}.csv", index=False
)
(RUN / f"epoch3_test_metrics_{stamp}.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n\nTEST FINISHED")
print(json.dumps(result, ensure_ascii=False, indent=2))

Translated 150/150

TEST FINISHED
{
  "system": "New Model2 run — recovered epoch 3",
  "split": "test",
  "checkpoint": "/content/drive/MyDrive/NMT_Sports_Project/models/Model2_epoch_extension_NEW_20260923_135303/epoch_3_weights",
  "BLEU": 43.70022781026532,
  "chrF++": 66.96248729035986,
  "generation_seconds": 16.30350351333618,
  "num_beams": 4,
  "max_length": 128,
  "input_max_length": 512,
  "bleu_signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
  "chrf_signature": "nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|version:2.6.0"
}


In [14]:
%pip -q install "sentence-transformers==5.1.2" "transformers==4.57.1" scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 32.8 MB/s eta 0:00:00


In [15]:
import torch
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cuda" if torch.cuda.is_available() else "cpu",
    cache_folder="/content/sbert_cache",
)

with torch.inference_mode():
    embeddings = sbert.encode(
        ["بازیکن گل زد.", "The player scored a goal."],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

print("Sentence-BERT آماده است ✅")
print("ابعاد بردارها:", embeddings.shape)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT آماده است ✅
ابعاد بردارها: (2, 384)


In [16]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/NMT_Sports_Project")
train_df = pd.read_parquet(BASE / "data/splits/train.parquet")
val_df = pd.read_parquet(BASE / "data/splits/validation.parquet")

assert len(train_df) == 1023
assert len(val_df) == 150
assert "sbert" in globals(), "ابتدا سلول بارگذاری Sentence-BERT را اجرا کن."

print("Train:", len(train_df), "| Validation:", len(val_df))
print("Sentence-BERT آماده است ✅")

row = val_df.iloc[0]
print("\nSOURCE:\n", row["source"])

for k in (1, 2, 3):
    print(f"\nCONTEXT {k}:")
    print(str(row[f"context_{k}_text"])[:1200])

    col = f"context_{k}_segments_json"
    if col in val_df.columns:
        print("SEGMENTS:")
        print(str(row[col])[:1200])

Train: 1023 | Validation: 150
Sentence-BERT آماده است ✅

SOURCE:
 پس‌تحلیل بازی به بهبود استراتژی‌ها کمک می‌کند.

CONTEXT 1:
پس‌تحلیل بازی به بهبود استراتژی‌ها کمک می‌کند و دقت پاس‌ها بالا بود، در حالی که جریان بازی پس از ضدحمله عوض شد و بازی بدون گل دو تیم نشان‌دهنده ضعف خط حمله و تمرکز بر دفاع بود. استقامت پایه اصلی ورزشکاران ماراتنی است و تناسب‌اندام یکی از اهداف اصلی ورزش محسوب می‌شود، در حالی که ذهنیت برنده‌نما هرگز در برابر سختی‌ها تسلیم نمی‌شود و ریکاوری ذهنی برای بازگشت به فرم اصلی ضروری است. تمرین سخت تنها راه رسیدن به قله است و استفاده از GPS در تمرینات برای سنجش مسافت طی‌شده الزامی است، در حالی که تجمع اسید لاکتیک در عضله باعث خستگی زودرس شد و مقاومت هوا در سرعت‌های بالا تأثیر قابل‌توجهی دارد. حمل مشعل المپیک برای هر ورزشکاری یک افتخار است و مشعل المپیک نماد صلح و ورزش است، در حالی که تالار مشاهیر از قهرمان تقدیر کرد و شکستن رکورد جهانی در رشته دو و میدانی همه را شگفت‌زده کرد. دروازه‌بان تیم کلین‌شیت حفظ کرد و گل تیم در دقیقه ۷۵ زده شد، در حالی که پرش با اسب در مسابقات سوارک

In [17]:
import re, unicodedata
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# پروتکل مشترک و ثابت برای هر دو روش
ALPHA, BETA, GAMMA, TAU = 1.0, 0.35, 0.10, 0.02
WORD_BUDGET = 180

def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(str.maketrans({"ي": "ی", "ك": "ک"}))
    text = re.sub(r"[\u200c\u200d\u200e\u200f]", " ", text)
    text = re.sub(r"[-‐‑‒–—−]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def echo_key(text):
    text = normalize_text(text).lower()
    return re.sub(r"[^\w\s]", "", text).strip()

def candidates_for(row):
    source_key = echo_key(row["source"])
    candidates, seen = [], set()

    for k in (1, 2, 3):
        value = row[f"context_{k}_text"]
        if pd.isna(value):
            continue

        # تقسیم یکسان برای هر دو روش
        parts = re.split(
            r"<segment_sep>|[.!?؟؛\n]+", str(value)
        )
        for part in parts:
            text = normalize_text(part)
            key = echo_key(text)

            if not key or key in seen:
                continue
            seen.add(key)

            # حذف جمله‌ای که متن کامل مبدأ را در خود دارد
            if source_key and f" {source_key} " in f" {key} ":
                continue

            if len(text.split()) <= WORD_BUDGET:
                candidates.append(text)

    return candidates

train_candidates = [
    candidates_for(row) for _, row in train_df.iterrows()
]
val_candidates = [
    candidates_for(row) for _, row in val_df.iterrows()
]

# برازش TF-IDF فقط روی متن مبدأ و contextهای TRAIN
fit_texts = sorted(set(
    [normalize_text(s) for s in train_df["source"]]
    + [c for group in train_candidates for c in group]
))

tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=50000,
    sublinear_tf=True,
)
tfidf.fit(fit_texts)

# بردارسازی متون Validation؛ بدون استفاده از target
texts = sorted(set(
    [normalize_text(s) for s in val_df["source"]]
    + [c for group in val_candidates for c in group]
))
text_index = {text: i for i, text in enumerate(texts)}

# جلوگیری از برش خاموش متن‌های طولانی در Sentence-BERT:
# تقسیم توکنی، سپس میانگین وزنی بردار قطعات
token_budget = (
    sbert.max_seq_length
    - sbert.tokenizer.num_special_tokens_to_add(pair=False)
)

chunks, owners, weights = [], [], []
for i, text in enumerate(texts):
    ids = sbert.tokenizer.encode(text, add_special_tokens=False)
    for start in range(0, len(ids), token_budget):
        part = ids[start:start + token_budget]
        chunks.append(sbert.tokenizer.decode(
            part, skip_special_tokens=True
        ))
        owners.append(i)
        weights.append(len(part))

chunk_vectors = sbert.encode(
    chunks,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

semantic = np.zeros(
    (len(texts), chunk_vectors.shape[1]), dtype=np.float32
)
for owner, weight, vector in zip(owners, weights, chunk_vectors):
    semantic[owner] += weight * vector

semantic /= np.maximum(
    np.linalg.norm(semantic, axis=1, keepdims=True), 1e-12
)
lexical = tfidf.transform(texts)

def select_context(source, candidates, method):
    if not candidates:
        return ""

    ids = [text_index[normalize_text(source)]]
    ids += [text_index[c] for c in candidates]

    if method == "tfidf":
        matrix = lexical[ids]
        similarities = (matrix @ matrix.T).toarray()
    else:
        matrix = semantic[ids]
        similarities = np.clip(matrix @ matrix.T, -1, 1)

    relevance = similarities[0, 1:]
    lengths = np.array([len(c.split()) for c in candidates])
    selected, remaining = [], list(range(len(candidates)))
    words_used = 0

    while remaining:
        scored = []
        for j in remaining:
            if words_used + lengths[j] > WORD_BUDGET:
                continue

            redundancy = (
                max(0.0, max(
                    similarities[j + 1, k + 1] for k in selected
                ))
                if selected else 0.0
            )
            utility = (
                ALPHA * relevance[j]
                - BETA * redundancy
                - GAMMA * lengths[j] / WORD_BUDGET
            )
            scored.append((float(utility), j))

        if not scored:
            break

        utility, j = max(scored, key=lambda x: (x[0], -x[1]))
        if utility <= TAU:
            break

        selected.append(j)
        remaining.remove(j)
        words_used += lengths[j]

    # ترتیب متن اصلی برای هر دو روش
    return " ".join(candidates[j] for j in sorted(selected))

comparison_val = val_df.copy()

for method in ("tfidf", "sbert"):
    comparison_val[f"context_{method}"] = [
        select_context(row["source"], candidates, method)
        for (_, row), candidates
        in zip(val_df.iterrows(), val_candidates)
    ]

print("\nانتخاب context تمام شد ✅")
for method in ("tfidf", "sbert"):
    col = comparison_val[f"context_{method}"]
    print(
        method,
        "| nonempty:", int(col.ne("").sum()),
        "| mean words:", round(col.str.split().str.len().mean(), 2)
    )

print("\nTF-IDF نمونه اول:")
print(comparison_val.iloc[0]["context_tfidf"])
print("\nSentence-BERT نمونه اول:")
print(comparison_val.iloc[0]["context_sbert"])

Batches:   0%|          | 0/10 [00:00<?, ?it/s]


انتخاب context تمام شد ✅
tfidf | nonempty: 149 | mean words: 88.23
sbert | nonempty: 150 | mean words: 171.2

TF-IDF نمونه اول:
در چنین شرایطی، نشست خبری طوفانی پس از باخت تلخ برگزار می شود و مصاحبه مربی بعد از بازی ابعاد جدیدی را روشن می کند با این حال، آینده باشگاه با جوانان روشن است و استراتژی هایی چون صادرات ورزشکار به لیگ خارجی، انتقال بازیکن با ارز خارجی و توجه به این نکته که آینده ورزش کشور در گرو توجه به نساجی و تولید داخلی است، راهگشا خواهد بود پس تحلیل بازی نه تنها به بهبود استراتژی ها کمک می کند، بلکه در شناسایی نقاط ضعف و قوت تیم مؤثر است

Sentence-BERT نمونه اول:
بازی منصفانه و کرامت انسانی از اصول اولیه ورزش حرفه ای است و سنت جوانمردی در فیرپلی رعایت شد، در حالی که جریمه باخت سنگین در قرارداد بازیکنان باعث افزایش جنگندگی شد و انتقال بازیکن با ارز خارجی انجام شد با این حال، حواشی مدیریتی و ساختاری گاهی سایه بر ورزش می اندازد پس تحلیل بازی نه تنها به بهبود استراتژی ها کمک می کند، بلکه در شناسایی نقاط ضعف و قوت تیم مؤثر است با این حال، گل تیم در دقیقه ۷۵ و ضدحمله ای هوشمندا

In [18]:
import json, time
from datetime import datetime
from pathlib import Path
import torch
from sacrebleu.metrics import BLEU, CHRF

assert "net" in globals() and "tok" in globals(), (
    "مدل ترجمه در حافظه نیست؛ توقف کن و همین پیام را بفرست."
)
assert "comparison_val" in globals()
assert len(comparison_val) == 150
assert comparison_val["target"].notna().all()

BASE = Path("/content/drive/MyDrive/NMT_Sports_Project")
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = BASE / "metrics" / f"context_comparison_validation_{stamp}"
OUT.mkdir(parents=True, exist_ok=False)

protocol = {
    "experiment": "Reconstructed selector; inference-time context comparison",
    "split": "validation",
    "translation_checkpoint": str(getattr(net.config, "_name_or_path", "")),
    "sentence_encoder": (
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    ),
    "alpha": ALPHA, "beta": BETA, "gamma": GAMMA, "tau": TAU,
    "context_word_budget": WORD_BUDGET,
    "input_max_length": 512,
    "output_max_length": 128,
    "num_beams": 4,
    "do_sample": False,
    "batch_size": 4,
    "packing": "Original source, newline, selected context; right truncation",
    "reference": "Original validation target text",
    "limitation": (
        "Source-only-trained translation model; reconstructed selector; "
        "exact source-echo suppression does not remove paraphrases."
    ),
}
(OUT / "protocol.json").write_text(
    json.dumps(protocol, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

sources = comparison_val["source"].astype(str).tolist()
refs = comparison_val["target"].astype(str).tolist()

conditions = {
    "source_only": sources,
    "tfidf": [
        s + ("\n" + c if c else "")
        for s, c in zip(sources, comparison_val["context_tfidf"])
    ],
    "sbert": [
        s + ("\n" + c if c else "")
        for s, c in zip(sources, comparison_val["context_sbert"])
    ],
}

tok.src_lang = "pes_Arab"
tok.tgt_lang = "eng_Latn"
tok.truncation_side = "right"

source_lengths = [
    len(ids) for ids in tok(
        sources, truncation=False, padding=False
    )["input_ids"]
]
assert max(source_lengths) <= 512, "یک متن مبدأ از بودجه ورودی بلندتر است."

target_id = tok.convert_tokens_to_ids("eng_Latn")
assert target_id is not None and target_id != tok.unk_token_id

net.eval()
net.generation_config.max_new_tokens = None
device = next(net.parameters()).device
bleu, chrf = BLEU(), CHRF(word_order=2)
output_df = comparison_val.copy()
rows = []

for name, texts in conditions.items():
    lengths = [
        len(ids) for ids in tok(
            texts, truncation=False, padding=False
        )["input_ids"]
    ]
    predictions = []

    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        for i in range(0, len(texts), 4):
            batch = tok(
                texts[i:i+4],
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512,
            ).to(device)

            generated = net.generate(
                **batch,
                forced_bos_token_id=target_id,
                num_beams=4,
                do_sample=False,
                max_length=128,
                use_cache=True,
            )
            predictions.extend(tok.batch_decode(
                generated, skip_special_tokens=True
            ))

            if (i + 4) % 40 == 0 or i + 4 >= len(texts):
                print(
                    f"{name}: {min(i+4, len(texts))}/{len(texts)}",
                    flush=True,
                )

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    assert len(predictions) == len(refs)
    output_df[f"prediction_{name}"] = predictions
    output_df[f"input_tokens_{name}"] = lengths

    row = {
        "system": name,
        "split": "validation",
        "BLEU": bleu.corpus_score(predictions, [refs]).score,
        "chrF++": chrf.corpus_score(predictions, [refs]).score,
        "seconds": elapsed,
        "truncated_inputs": sum(n > 512 for n in lengths),
        "mean_input_tokens_before_truncation": sum(lengths) / len(lengths),
        "bleu_signature": str(bleu.get_signature()),
        "chrf_signature": str(chrf.get_signature()),
    }
    rows.append(row)

    # ذخیره پس از هر حالت برای حفظ خروجی‌های تکمیل‌شده
    output_df.to_csv(OUT / "validation_predictions.csv", index=False)
    pd.DataFrame(rows).to_csv(OUT / "validation_metrics.csv", index=False)
    print(
        f"{name}: BLEU={row['BLEU']:.4f} | "
        f"chrF++={row['chrF++']:.4f} | "
        f"truncated={row['truncated_inputs']}",
        flush=True,
    )

print("\nVALIDATION FINISHED ✅")
print(pd.DataFrame(rows)[
    ["system", "BLEU", "chrF++", "truncated_inputs"]
].to_string(index=False))
print("\nSaved to:", OUT)

source_only: 40/150
source_only: 80/150
source_only: 120/150
source_only: 150/150
source_only: BLEU=50.2223 | chrF++=69.5987 | truncated=0
tfidf: 40/150
tfidf: 80/150
tfidf: 120/150
tfidf: 150/150
tfidf: BLEU=4.8771 | chrF++=28.5597 | truncated=0
sbert: 40/150
sbert: 80/150
sbert: 120/150
sbert: 150/150
sbert: BLEU=2.7308 | chrF++=20.9182 | truncated=0

VALIDATION FINISHED ✅
     system      BLEU    chrF++  truncated_inputs
source_only 50.222346 69.598740                 0
      tfidf  4.877086 28.559675                 0
      sbert  2.730829 20.918228                 0

Saved to: /content/drive/MyDrive/NMT_Sports_Project/metrics/context_comparison_validation_20260923_193100


In [19]:
import shutil
from google.colab import files

folder = (
    "/content/drive/MyDrive/NMT_Sports_Project/metrics/"
    "context_comparison_validation_20260923_193100"
)
archive = shutil.make_archive(
    "/content/context_comparison_validation_backup",
    "zip",
    folder,
)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>